In [ ]:
# Get the Studio Ghibli Style Images Dataset... and Unzip it

!wget https://github.com/TachibanaYoshino/AnimeGANv2/releases/download/1.0/Hayao.tar.gz
!tar -xzvf Hayao.tar.gz

In [ ]:
import os
import pandas
import torch

from transformers import AutoTokenizer, BlipProcessor, BlipForConditionalGeneration
from PIL          import Image



# Load BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/blip-image-captioning-base")

# Avoid thse words unwanted words in the captions
# We want these features to be represented in the training image dataset itself
unwanted_words     = ["cartoon", "drawing", "painting", "animated", "anime"] #  + ["detective", "legend"] - generated too often?
unwanted_words_ids = [tokenizer.encode(unwanted_word, add_special_tokens=False) for unwanted_word in unwanted_words]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Caption generation function
def generate_captions(image_path):
    image  = Image.open(image_path).convert("RGB")
    inputs = processor(image, return_tensors="pt").to(device)

    output = model.generate(**inputs, bad_words_ids=unwanted_words_ids)

    return tokenizer.decode(output[0], skip_special_tokens=True)


IMAGE_DATASET_FOLDER = "./Hayao/style"

image_paths = []
for filename in os.listdir(IMAGE_DATASET_FOLDER):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_paths.append(os.path.join(IMAGE_DATASET_FOLDER, filename))

# Generate captions for all images
captions = []
for img_path in image_paths:
    try:
        caption = generate_captions(img_path)
        print(f"{os.path.basename(img_path)}: {caption}") # Logging
        captions.append({"filename": os.path.basename(img_path), "caption": caption})
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

# Save results to a CSV File
df = pandas.DataFrame(captions)
df.to_csv("image_captions.csv", index=False)

print("Captions saved to image_captions.csv!")

